# 🫁 Pneumonia Detection – Master Execution Notebook

This notebook runs the complete Pneumonia Detection pipeline on **Google Colab (GPU accelerated)** or on **your local machine (CPU/GPU)**.  
The environment is auto‑detected, and paths/instructions adjust accordingly.

---

## ⚡ Architecture Overview
- **Google Colab (GPU Runtime)**  
  Recommended for training models.  
  Example: the **DenseNet121 + Swin Ensemble (CheX‑DS)** trains in ~38 minutes on GPU.  
  - Mounts Google Drive to persist weights.  
  - Saves dataset to local scratch disk (`/content/data/`) to bypass Drive latency.

- **Local Machine (CPU/GPU)**  
  Best for evaluation, inference, and running the **Streamlit app**.  
  Re‑uses weights generated on Colab.

---

## Google Colab Setup
1. **Enable GPU**  
   - Go to **Runtime → Change runtime type → GPU (T4, L4, or A100)** → **Save**.

2. **Upload Project Folder**  
   - Upload the `pneumonia-detection-cnn-xray` folder directly under **My Drive/** in Google Drive.  
   - ⚠️ **Important:** Exclude the `data/` and `venv/` directories when uploading.  
     This keeps the upload size minimal and completes in seconds.


## Environment Detection & Google Drive Mount
Detects if the code is running on Google Colab or a local machine. If running on Colab, it mounts Google Drive.

In [2]:
import sys
import os

# Detect environment
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("📍 Google Colab environment detected. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("🖥️ Local machine detected. Skipping Google Drive mount.")

📍 Google Colab environment detected. Mounting Google Drive...
Mounted at /content/drive


## Set Workspace Directory
Sets the current working directory to the project root directory.

In [3]:
if IS_COLAB:
    project_dir = '/content/drive/MyDrive/pneumonia-detection-cnn-xray'
    if not os.path.exists(project_dir):
        print(f"❌ ERROR: Project folder '{project_dir}' was not found in your Google Drive.")
        print("Please verify that you uploaded 'pneumonia-detection-cnn-xray' directly under 'My Drive'.")
    else:
        %cd {project_dir}
        print("\n📁 Google Drive Workspace Directory Contents:")
        !ls -la
else:
    print("🖥️ Running locally. Current working directory:")
    %pwd
    print("\n📁 Local Directory Contents:")
    print(os.listdir('.'))

/content/drive/MyDrive/pneumonia-detection-cnn-xray

📁 Google Drive Workspace Directory Contents:
total 1477
-rw------- 1 root root  13993 Jun 27 01:03  app.py
-rw------- 1 root root 981887 Jun 27 01:03  CheX-DS.pdf
-rw------- 1 root root  23304 Jun 27 01:03 'Confusion Matrix CheX-DS.png'
-rw------- 1 root root 403681 Jun 27 01:03  demo_screenshot.png
-rw------- 1 root root    591 Jun 27 01:03  Dockerfile
-rw------- 1 root root    159 Jun 27 01:03  .dockerignore
-rw------- 1 root root   2909 Jun 27 01:03  download_data.py
-rw------- 1 root root    207 Jun 27 01:03  .gitignore
drwx------ 2 root root   4096 Jun 27 01:04  .idea
-rw------- 1 root root   4201 Jun 27 01:03  inference.py
-rw------- 1 root root   1346 Jun 27 01:03  main.py
drwx------ 2 root root   4096 Jun 27 01:04  models
drwx------ 2 root root   4096 Jun 27 01:04  notebooks
-rw------- 1 root root   8582 Jun 27 01:03  README.md
-rw------- 1 root root   1018 Jun 27 01:03  requirements.txt
drwx------ 2 root root   4096 Jun 27 0

## Hardware Verification & Package Installation
Verifies hardware runtime support (GPU vs CPU) and installs required Python libraries.

In [4]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"GPU (CUDA) Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    if IS_COLAB:
        print("⚠️ WARNING: GPU not detected. Training will be extremely slow. Enable GPU in Runtime -> Change runtime type.")
    else:
        print("ℹ️ Running on CPU (standard for local development/inference).")

PyTorch version: 2.11.0+cu128
GPU (CUDA) Available: True
GPU Device Name: Tesla T4


In [5]:
# Install project dependencies
import sys, subprocess

def pip_install(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs)

try:
    import google.colab
    print("📍 Google Colab detected.\nInstalling auxiliary packages (Streamlit, GitPython)...\n")
    pip_install(["streamlit", "GitPython"])
    print("✅ Done (no runtime restart required).")
except ImportError:
    print("🖥️ Local environment detected.\nInstalling dependencies from requirements.txt...\n")
    pip_install(["-r", "requirements.txt"])
    print("✅ Done.")

📍 Google Colab detected.
Installing auxiliary packages (Streamlit, GitPython)...

✅ Done (no runtime restart required).


### Verify Model Architecture Compilation
Verifies that all model architectures build correctly on the active device.

In [6]:
!python test_model.py

Using device: cuda
CNN Output Shape: torch.Size([1, 2])
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 179MB/s]
ResNet Output Shape: torch.Size([1, 2])
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100% 30.8M/30.8M [00:00<00:00, 141MB/s]
Downloading: "https://download.pytorch.org/models/swin_b-68c6b09e.pth" to /root/.cache/torch/hub/checkpoints/swin_b-68c6b09e.pth
100% 335M/335M [00:01<00:00, 184MB/s]
CheXDS Output Shape: torch.Size([1, 2])
Models built successfully.


## High-Performance Data Extraction
Downloads the dataset (if missing) and extracts it. Checks file size integrity using S3 HTTP HEAD values to ensure incomplete downloads are overwritten.

In [7]:
# Run the download and extraction script
!python download_data.py

Downloading: 100%|#############################################| 1.14G/1.14G [00:30<00:00, 40.2MB/s]
Download complete.
Extracting /content/data/xray_dataset.tar.gz...
Extracting:   0% 0/5867 [00:00<?, ?files/s]/content/drive/MyDrive/pneumonia-detection-cnn-xray/download_data.py:74: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=output_path)
Extracting: 100% 5867/5867 [00:12<00:00, 472.42files/s] 
Extraction complete.


In [8]:
# Verify that data files were downloaded
import os
from src.config import DATA_DIR

print(f"Checking dataset structure at: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    print("Dataset folder structure:", os.listdir(DATA_DIR))
    train_path = os.path.join(DATA_DIR, 'train')
    if os.path.exists(train_path):
        print("Train subfolders:", os.listdir(train_path))
else:
    print("❌ Dataset folder not found. Please run download_data.py above.")

Using device: cuda
Checking dataset structure at: /content/data/chest_xray
Dataset folder structure: ['test', '.DS_Store', 'train']
Train subfolders: ['NORMAL', '.DS_Store', 'PNEUMONIA']


### Verify Data Loading & Dataset Pipeline
Runs a pre-flight test verifying data loaders can fetch a batch of scans.

In [9]:
!python test_data.py

Using device: cuda
Loading data from: /content/data/chest_xray/train
Stats: 4186 Train, 1046 Val, 624 Test images.
Classes: ['NORMAL', 'PNEUMONIA']
Batch shape: torch.Size([32, 3, 256, 256])
Labels shape: torch.Size([32])
Data Loading Successful


## Train Models
Runs the training pipelines. Weight files (`pneumonia_model_{model}.pth`) are saved directly to `models/`.

### Option A: Train a Single Model
Run one of the cells below to train a specific model architecture.

In [9]:
# Train baseline SimpleCNN model
!python main.py --model cnn

Using device: cuda
Initializing Data Loaders...
Loading data from: /content/data/chest_xray/train
Stats: 4186 Train, 1046 Val, 624 Test images.
Classes: ['NORMAL', 'PNEUMONIA']
Building CNN Model...

--- Starting Training: CNN on cuda ---
Epoch 1/15
Training: 100% 131/131 [01:21<00:00,  1.60batch/s, loss=0.469]
Train Loss: 0.4688 | Acc: 0.7774 | Val Loss: 0.2076 | Acc: 0.9369
Epoch 2/15
Training: 100% 131/131 [01:20<00:00,  1.63batch/s, loss=0.23]
Train Loss: 0.2300 | Acc: 0.9173 | Val Loss: 0.1210 | Acc: 0.9541
Epoch 3/15
Training: 100% 131/131 [01:21<00:00,  1.62batch/s, loss=0.166]
Train Loss: 0.1658 | Acc: 0.9439 | Val Loss: 0.0988 | Acc: 0.9646
Epoch 4/15
Training: 100% 131/131 [01:20<00:00,  1.63batch/s, loss=0.156]
Train Loss: 0.1565 | Acc: 0.9501 | Val Loss: 0.1231 | Acc: 0.9522
EarlyStopping counter: 1 out of 3
Epoch 5/15
Training: 100% 131/131 [01:19<00:00,  1.65batch/s, loss=0.143]
Train Loss: 0.1429 | Acc: 0.9546 | Val Loss: 0.1100 | Acc: 0.9579
EarlyStopping counter: 2 out

In [10]:
# Train Transfer Learning (ResNet50) model
!python main.py --model resnet

Using device: cuda
Initializing Data Loaders...
Loading data from: /content/data/chest_xray/train
Stats: 4186 Train, 1046 Val, 624 Test images.
Classes: ['NORMAL', 'PNEUMONIA']
Building RESNET Model...

--- Starting Training: RESNET on cuda ---
Epoch 1/10
Training: 100% 131/131 [01:26<00:00,  1.52batch/s, loss=0.266]
Train Loss: 0.2657 | Acc: 0.8985 | Val Loss: 0.1707 | Acc: 0.9570
Epoch 2/10
Training: 100% 131/131 [01:26<00:00,  1.52batch/s, loss=0.146]
Train Loss: 0.1463 | Acc: 0.9439 | Val Loss: 0.3459 | Acc: 0.9436
EarlyStopping counter: 1 out of 3
Epoch 3/10
Training: 100% 131/131 [01:27<00:00,  1.50batch/s, loss=0.0953]
Train Loss: 0.0953 | Acc: 0.9639 | Val Loss: 0.1448 | Acc: 0.9618
Epoch 4/10
Training: 100% 131/131 [01:27<00:00,  1.50batch/s, loss=0.0756]
Train Loss: 0.0756 | Acc: 0.9709 | Val Loss: 0.1247 | Acc: 0.9618
Epoch 5/10
Training: 100% 131/131 [01:27<00:00,  1.50batch/s, loss=0.0855]
Train Loss: 0.0855 | Acc: 0.9709 | Val Loss: 0.1252 | Acc: 0.9551
EarlyStopping coun

In [11]:
# Train CheX-DS (DenseNet + Swin Transformer Ensemble - Recommended)
!python main.py --model chexds

Using device: cuda
Initializing Data Loaders...
Loading data from: /content/data/chest_xray/train
Stats: 4186 Train, 1046 Val, 624 Test images.
Classes: ['NORMAL', 'PNEUMONIA']
Building CHEXDS Model...
Calculating class ratios for CheX-DS Loss...
Class Ratios (rho): [0.25609174370765686, 0.7439082860946655]

--- Starting Training: CHEXDS on cuda ---
Epoch 1/10
Training: 100% 131/131 [02:57<00:00,  1.36s/batch, loss=0.289]
Train Loss: 0.2892 | Acc: 0.8817 | Val Loss: 0.1845 | Acc: 0.9503
Epoch 2/10
Training: 100% 131/131 [03:00<00:00,  1.38s/batch, loss=0.15]
Train Loss: 0.1502 | Acc: 0.9434 | Val Loss: 0.1063 | Acc: 0.9532
Epoch 3/10
Training: 100% 131/131 [03:01<00:00,  1.38s/batch, loss=0.102]
Train Loss: 0.1022 | Acc: 0.9601 | Val Loss: 0.0843 | Acc: 0.9579
Epoch 4/10
Training: 100% 131/131 [03:02<00:00,  1.39s/batch, loss=0.103]
Train Loss: 0.1030 | Acc: 0.9625 | Val Loss: 0.2161 | Acc: 0.9159
EarlyStopping counter: 1 out of 3
Epoch 5/10
Training: 100% 131/131 [03:00<00:00,  1.38s/

### Option B: Run Full Benchmarks (All Models)
Trains all models sequentially on the GPU, logging the results in `results/experiments_summary.json`.

In [ ]:
!python run_experiments.py

## Evaluation & Result Visualization
Runs evaluation and generates classification reports and Confusion Matrix/ROC curve plots in `results/`.

In [12]:
%matplotlib inline
# Choose model to evaluate: cnn, resnet, or chexds
!python visualize_results.py --model chexds

Using device: cuda
Loading Test Data for Evaluation...
Loading data from: /content/data/chest_xray/train
Stats: 4186 Train, 1046 Val, 624 Test images.
Classes: ['NORMAL', 'PNEUMONIA']
Loading CHEXDS weights from /content/drive/MyDrive/pneumonia-detection-cnn-xray/models/pneumonia_model_chexds.pth...
Running Inference on Test Set (this may take a moment)...

FINAL CLASSIFICATION REPORT (CHEXDS)
              precision    recall  f1-score   support

      NORMAL       0.99      0.63      0.77       234
   PNEUMONIA       0.82      1.00      0.90       390

    accuracy                           0.86       624
   macro avg       0.91      0.81      0.83       624
weighted avg       0.88      0.86      0.85       624

Saved confusion matrix plot to: /content/drive/MyDrive/pneumonia-detection-cnn-xray/results/confusion_matrix_chexds.png
Figure(800x600)
Saved ROC curve plot to: /content/drive/MyDrive/pneumonia-detection-cnn-xray/results/roc_curve_chexds.png
Figure(800x600)


## Run Streamlit Web Application inside Colab (Optional)
Exposes the Streamlit interface using `localtunnel` port forwarding. **Skip this step if running locally.**

In [13]:
if IS_COLAB:
    # 1. Install localtunnel
    !npm install -g localtunnel
else:
    print("🖥️ Running locally. Skip localtunnel setup and run Step 8 directly.")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [12]:
!npm fund

⠙pneumonia-detection-cnn-xray

⠙

In [15]:
if IS_COLAB:
    # 2. Find your Colab public IP address (used as the password for localtunnel verification)
    import urllib
    print("Your Colab Endpoint Password:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
else:
    print("🖥️ Running locally.")

Your Colab Endpoint Password: 34.125.218.27


In [14]:
if IS_COLAB:
    # 3. Run Streamlit with localtunnel
    # Click the URL generated below, enter the 'Endpoint Password', and click submit.
    !streamlit run app.py & npx localtunnel --port 8501
else:
    print("🖥️ Running locally. Run command 'streamlit run app.py'.")

⠙⠹⠸⠼⠴⠦⠧

your url is: https://wicked-pianos-boil.loca.lt
2026-06-28 19:48:42.046 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.141.151.143:8501

Using device: cpu
  Stopping...
^C


## Run Streamlit App Locally (Recommended)
If you trained your models on Google Colab, download the generated weight files (`pneumonia_model_*.pth`) from your Google Drive into your local `models/` directory.

To start the Streamlit web dashboard locally on your PC:
1. Open a terminal in your project root directory.
2. Run the application:
   ```bash
   streamlit run app.py
   ```
3. Open `http://localhost:8501` in your web browser.
4. Choose between `CheX-DS`, `ResNet50`, and `Simple CNN` architectures dynamically in the sidebar dropdown!